# Fase 2: Preprocesamiento, Calidad y Limpieza de Datos
**Proyecto:** Análisis del desempeño SIMCE 2025 (4° Básico - Matemática)

## Objetivo de la Fase 2
Construir un pipeline reproducible que cargue la base bruta, estandarice tipos de datos, identifique y documente los registros **Efectivos** y **No Efectivos**, enriquezca la información geográfica y administrativa, aplique validaciones de calidad y genere un archivo procesado apto para la Fase 3.

Esta versión agrega explícitamente una **trazabilidad de los datos filtrados**, con especial atención a los registros No Efectivos. La exclusión no se trata como una eliminación silenciosa: antes de filtrar se cuantifica cuántos registros se descartan, por qué se descartan, cuántos conservan puntaje y qué tan material es su exclusión.

### Contenido del Pipeline F2
1. Configuración del entorno y rutas dinámicas.
2. Definición de diccionarios y mapeos.
3. Extracción de datos crudos.
4. Coerción y estandarización de tipos.
5. Diagnóstico de efectividad, causas de exclusión y materialidad.
6. Filtrado de registros No Efectivos.
7. Enriquecimiento geográfico y taxonómico.
8. Renombrado y estandarización de columnas.
9. Validaciones de calidad.
10. Exportación del dataset procesado.


In [ ]:
import sys
from pathlib import Path
from datetime import datetime
import pandas as pd

try:
    from zoneinfo import ZoneInfo
except ImportError:
    ZoneInfo = None

# 1. DETECCIÓN DINÁMICA DE LA RAÍZ DEL REPOSITORIO
def obtener_raiz_repositorio(directorio_actual: Path = Path.cwd()) -> Path:
    directorio = directorio_actual.resolve()
    for parent in [directorio] + list(directorio.parents):
        if (parent / ".git").exists() or (parent / "requirements.txt").exists():
            return parent
    return directorio_actual.resolve()

RAIZ = obtener_raiz_repositorio()
RAW_FILE = RAIZ / "data" / "raw" / "simce4b2025_rbd_final.csv"
OUTPUT_DIR = RAIZ / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Nombre de salida con fecha y hora de ejecución: AAAAMMDDHHMM
if ZoneInfo is not None:
    try:
        FECHA_EJECUCION = datetime.now(ZoneInfo("America/Santiago"))
    except Exception:
        FECHA_EJECUCION = datetime.now().astimezone()
else:
    FECHA_EJECUCION = datetime.now().astimezone()

MARCA_EJECUCION = FECHA_EJECUCION.strftime("%Y%m%d%H%M")
OUTPUT_FILE = OUTPUT_DIR / f"simce4b2025_matematica_efectiva_{MARCA_EJECUCION}.csv"

print(f"Directorio raíz: {RAIZ}")
print(f"Archivo de entrada: {RAW_FILE}")
print(f"Archivo de salida: {OUTPUT_FILE.name}")

assert RAW_FILE.exists(), f"Error: No se encuentra el archivo de entrada en {RAW_FILE}"


## 2. Diccionarios de Categorías y Mapeos Geográficos
Definición de las estructuras auxiliares para la codificación y categorización de dependencias administrativas, grupos socioeconómicos (GSE), zonas de ruralidad y ordenamiento macrozonal de Chile (Norte a Sur).

In [ ]:
# Mapeo de Administración Escolar
DEPENDENCIA_1 = {
    1: "Municipal Corporación",
    2: "Municipal DAEM",
    3: "Particular subvencionado",
    4: "Particular pagado",
    5: "Corporación de administración delegada",
    6: "Servicio Local de Educación",
}

DEPENDENCIA_2 = {
    1: "Municipal",
    2: "Particular subvencionado",
    3: "Particular pagado",
    4: "Servicio Local de Educación",
}

# Mapeos Socioeconómicos y Entorno
GSE = {
    1: "Bajo",
    2: "Medio bajo",
    3: "Medio",
    4: "Medio alto",
    5: "Alto",
}

RURALIDAD = {
    1: "Urbano",
    2: "Rural",
}


# Glosas utilizadas para interpretar las observaciones del puntaje.
# Estas categorías explican por qué un registro puede no ser representativo
# o no ser reportable como resultado efectivo.
OBS_PUNTAJE = {
    1: "Cantidad de estudiantes evaluados insuficiente (6 o menos)",
    2: "Resultados no representativos por causas ajenas a la Agencia",
    3: "Resultados no representativos por causas ajenas al establecimiento",
    4: "Prueba extendida: no permite evaluar esta asignatura",
}

# Geografía y Macrozonas de Chile (Orden de Norte a Sur)
GEOGRAFIA_REGION = {
    15: ("Norte", "Norte Grande", 1),   # Arica y Parinacota
    1:  ("Norte", "Norte Grande", 2),   # Tarapacá
    2:  ("Norte", "Norte Grande", 3),   # Antofagasta
    3:  ("Norte", "Norte Chico",  4),   # Atacama
    4:  ("Norte", "Norte Chico",  5),   # Coquimbo
    5:  ("Centro", "Zona Central", 6),  # Valparaíso
    13: ("Centro", "Zona Central", 7),  # Metropolitana
    6:  ("Centro", "Zona Central", 8),  # O'Higgins
    7:  ("Centro", "Zona Central", 9),  # Maule
    16: ("Sur",    "Zona Sur",    10),  # Ñuble
    8:  ("Sur",    "Zona Sur",    11),  # Biobío
    9:  ("Sur",    "Zona Sur",    12),  # Araucanía
    14: ("Sur",    "Zona Sur",    13),  # Los Ríos
    10: ("Sur",    "Zona Sur",    14),  # Los Lagos
    11: ("Sur",    "Zona Austral", 15), # Aysén
    12: ("Sur",    "Zona Austral", 16), # Magallanes
}

print("Diccionarios y mapeos cargados correctamente en memoria.")


## 3. Extracción de Datos Crudos (Raw Load)
Carga selectiva de columnas necesarias desde el archivo plano original (`.csv` codificado en `CP1252` con separador `;`). Esto optimiza el uso de memoria RAM evitando la carga de variables prescindibles.

In [ ]:
COLUMNAS_INPUT = [
    "rbd", "dvrbd", "nom_rbd",
    "cod_reg_rbd", "nom_reg_rbd",
    "cod_pro_rbd", "nom_pro_rbd",
    "cod_com_rbd", "nom_com_rbd",
    "cod_deprov_rbd", "nom_deprov_rbd",
    "cod_depe1", "cod_depe2", "cod_grupo", "cod_rural_rbd",
    "nalu_mate4b_rbd", "prom_mate4b_rbd",
    "palu_eda_ins_mate4b_rbd", "palu_eda_ele_mate4b_rbd", "palu_eda_ade_mate4b_rbd",
    "marca_mate4b_rbd", "noaplica", "codigo_bbdd", "fecha_bbdd", "grado", "agno"
]

df_raw = pd.read_csv(
    RAW_FILE,
    sep=";",
    encoding="cp1252",
    usecols=COLUMNAS_INPUT,
    low_memory=False
)

print(f"Dimensiones preliminares: {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas.")
# Conservamos una referencia del bruto completo para comparar sus nulos con la salida.
df_bruto_completo = pd.read_csv(RAW_FILE, sep=";", encoding="cp1252", low_memory=False)


## 4. Coerción y Estandarización Numérica
Conversión explícita de campos a tipos de datos numéricos. Se conservan los nulos originales y se detiene la ejecución si aparecen valores no convertibles, para evitar perder información silenciosamente.

También convertimos `fecha_bbdd` desde AAAAMMDD a una fecha: por ejemplo, 20260622 pasa a 2026-06-22. Es una fecha de la base y es constante en este archivo; no representa distintas fechas de evaluación ni permite estudiar tendencias. El criterio del 60% de nulos se evalúa sobre el bruto y esta conversión no cambia ese resultado.

In [ ]:
columnas_numericas = [
    "rbd", "dvrbd", "cod_reg_rbd", "cod_pro_rbd", "cod_com_rbd",
    "cod_deprov_rbd", "cod_depe1", "cod_depe2", "cod_grupo", "cod_rural_rbd",
    "nalu_mate4b_rbd", "prom_mate4b_rbd", "palu_eda_ins_mate4b_rbd",
    "palu_eda_ele_mate4b_rbd", "palu_eda_ade_mate4b_rbd", "marca_mate4b_rbd",
    "noaplica", "agno"
]

for col in columnas_numericas:
    convertida = pd.to_numeric(df_raw[col], errors="coerce")
    invalidos = df_raw[col].notna() & convertida.isna()
    assert not invalidos.any(), f"Valores no numéricos en {col}: {int(invalidos.sum())}"
    df_raw[col] = convertida

print("Transformación a tipos numéricos completada.")

# La fecha tiene ocho dígitos, pero no es una cantidad: la convertimos por separado.
# El formato explícito evita interpretarla como un número de días o segundos.
fecha_texto = df_raw["fecha_bbdd"].astype("string").str.strip()
assert fecha_texto.str.fullmatch(r"[0-9]{8}").fillna(False).all(), "Revisar fecha_bbdd: se espera AAAAMMDD sin vacíos."
df_raw["fecha_bbdd"] = pd.to_datetime(fecha_texto, format="%Y%m%d", errors="raise")
assert df_raw["fecha_bbdd"].notna().all(), "Hay fechas de base ausentes."
print("Fecha de la base:", df_raw["fecha_bbdd"].dt.strftime("%Y-%m-%d").unique())


## 5. Diagnóstico de Efectividad, Registros Filtrados y Materialidad

Antes de eliminar cualquier fila se construye un diagnóstico completo del efecto del filtro.

### Criterio de efectividad
Un registro se considera **Efectivo** cuando cumple simultáneamente:

- `nalu_mate4b_rbd > 0`, es decir, existe al menos un estudiante evaluado.
- `marca_mate4b_rbd` no corresponde a una observación que invalide o limite la representatividad del resultado.

En caso contrario se clasifica como **No Efectivo**.

### ¿Por qué no basta con eliminar los No Efectivos?
Porque eliminar filas puede modificar la cobertura de la base y, eventualmente, el resultado de análisis posteriores. Por eso esta etapa mide:

1. cantidad y porcentaje de registros excluidos;
2. motivo principal de exclusión;
3. cantidad de alumnos asociados a los registros excluidos;
4. cantidad de registros excluidos que aun conservan puntaje;
5. peso de esos puntajes excluidos respecto del total de registros con puntaje;
6. diferencia entre el promedio de los Efectivos y el promedio hipotético al incluir todos los registros que tienen puntaje;
7. concentración territorial de los casos excluidos con puntaje.

### Materialidad
Para esta F2 se utiliza un **criterio interno y explícito**, no una norma oficial SIMCE:

- **Materialidad de cobertura:** se considera material si los registros excluidos representan al menos 5% del archivo bruto.
- **Materialidad directa sobre puntajes:** se considera material si los registros excluidos que poseen puntaje representan al menos 5% de los registros con puntaje.
- **Sensibilidad del promedio:** se informa el cambio porcentual del promedio al incorporar hipotéticamente los registros excluidos con puntaje. No se utilizan esos registros en la salida final; el cálculo solo sirve como prueba de sensibilidad.

Esta separación es importante: una exclusión puede ser material por cantidad de establecimientos y, al mismo tiempo, tener baja incidencia directa sobre la disponibilidad de puntajes.


In [ ]:
# ============================================================
# DIAGNÓSTICO PREVIO AL FILTRO
# ============================================================

MARCAS_EXCLUIDAS = list(OBS_PUNTAJE.keys())

# Glosa de observación, conservada para explicar el filtro.
df_raw["obs_puntaje"] = df_raw["marca_mate4b_rbd"].map(OBS_PUNTAJE)

condicion_alumnos = df_raw["nalu_mate4b_rbd"].fillna(0).gt(0)
condicion_sin_observacion = df_raw["obs_puntaje"].isna()
condicion_efectiva = condicion_alumnos & condicion_sin_observacion

df_raw["efectividad"] = "No Efectiva"
df_raw.loc[condicion_efectiva, "efectividad"] = "Efectiva"

# Motivo de exclusión. Se explicita también la intersección de causas.
sin_alumnos = ~condicion_alumnos
con_observacion = ~condicion_sin_observacion

df_raw["motivo_exclusion"] = "Registro efectivo"
df_raw.loc[sin_alumnos & ~con_observacion, "motivo_exclusion"] = "Sin alumnos evaluados"
df_raw.loc[~sin_alumnos & con_observacion, "motivo_exclusion"] = "Observación de puntaje"
df_raw.loc[sin_alumnos & con_observacion, "motivo_exclusion"] = "Sin alumnos y con observación de puntaje"

# Totales principales
total_registros = len(df_raw)
total_efectivos = int(condicion_efectiva.sum())
total_no_efectivos = int((~condicion_efectiva).sum())
pct_no_efectivos = total_no_efectivos / total_registros * 100

# Registros con puntaje
con_puntaje = df_raw["prom_mate4b_rbd"].notna()
no_efectivos_con_puntaje = (~condicion_efectiva) & con_puntaje
total_con_puntaje = int(con_puntaje.sum())
cantidad_no_efectivos_con_puntaje = int(no_efectivos_con_puntaje.sum())
pct_no_efectivos_sobre_con_puntaje = (
    cantidad_no_efectivos_con_puntaje / total_con_puntaje * 100
    if total_con_puntaje else 0.0
)

# Alumnos asociados. Se usa como dimensión de cobertura, no como sustituto de representatividad.
alumnos_total = float(df_raw["nalu_mate4b_rbd"].fillna(0).sum())
alumnos_no_efectivos = float(
    df_raw.loc[~condicion_efectiva, "nalu_mate4b_rbd"].fillna(0).sum()
)
pct_alumnos_no_efectivos = (
    alumnos_no_efectivos / alumnos_total * 100 if alumnos_total else 0.0
)

# Sensibilidad del promedio: comparación informativa, NO cambia el criterio de salida.
promedio_efectivos = df_raw.loc[condicion_efectiva, "prom_mate4b_rbd"].mean()
promedio_todos_con_puntaje = df_raw.loc[con_puntaje, "prom_mate4b_rbd"].mean()

diferencia_promedio_puntos = (
    float(promedio_todos_con_puntaje - promedio_efectivos)
    if pd.notna(promedio_efectivos) and pd.notna(promedio_todos_con_puntaje)
    else float("nan")
)

diferencia_promedio_pct = (
    abs(diferencia_promedio_puntos) / abs(promedio_efectivos) * 100
    if pd.notna(diferencia_promedio_puntos) and promedio_efectivos not in [0, None]
    else float("nan")
)

# Umbrales internos, declarados para que la decisión sea auditable.
UMBRAL_MATERIALIDAD = 5.0

material_cobertura = pct_no_efectivos >= UMBRAL_MATERIALIDAD
material_puntajes = pct_no_efectivos_sobre_con_puntaje >= UMBRAL_MATERIALIDAD

if material_cobertura and material_puntajes:
    conclusion_materialidad = (
        "MATERIAL en cobertura y también en disponibilidad directa de puntajes."
    )
elif material_cobertura and not material_puntajes:
    conclusion_materialidad = (
        "MATERIAL en cobertura, pero de BAJA MATERIALIDAD DIRECTA sobre la disponibilidad de puntajes. "
        "Debe revisarse igualmente si los casos excluidos se concentran en territorios o grupos específicos."
    )
elif not material_cobertura and material_puntajes:
    conclusion_materialidad = (
        "BAJA MATERIALIDAD en cobertura, pero MATERIAL entre los registros que poseen puntaje."
    )
else:
    conclusion_materialidad = (
        "BAJA MATERIALIDAD tanto en cobertura como en disponibilidad directa de puntajes."
    )

# Resumen principal
resumen_filtro = pd.DataFrame({
    "indicador": [
        "Registros originales",
        "Registros efectivos",
        "Registros no efectivos",
        "% registros no efectivos",
        "Registros con puntaje",
        "No efectivos con puntaje",
        "% no efectivos con puntaje / registros con puntaje",
        "Alumnos asociados al bruto",
        "Alumnos asociados a no efectivos",
        "% alumnos asociados a no efectivos",
        "Promedio puntaje - efectivos",
        "Promedio puntaje - todos con puntaje",
        "Diferencia de promedio (puntos)",
        "Diferencia relativa del promedio (%)",
    ],
    "valor": [
        total_registros,
        total_efectivos,
        total_no_efectivos,
        round(pct_no_efectivos, 4),
        total_con_puntaje,
        cantidad_no_efectivos_con_puntaje,
        round(pct_no_efectivos_sobre_con_puntaje, 4),
        alumnos_total,
        alumnos_no_efectivos,
        round(pct_alumnos_no_efectivos, 4),
        round(float(promedio_efectivos), 4) if pd.notna(promedio_efectivos) else pd.NA,
        round(float(promedio_todos_con_puntaje), 4) if pd.notna(promedio_todos_con_puntaje) else pd.NA,
        round(diferencia_promedio_puntos, 4) if pd.notna(diferencia_promedio_puntos) else pd.NA,
        round(diferencia_promedio_pct, 4) if pd.notna(diferencia_promedio_pct) else pd.NA,
    ]
})

display(resumen_filtro)

# Distribución por motivo
resumen_motivos = (
    df_raw.loc[~condicion_efectiva]
    .groupby("motivo_exclusion", dropna=False)
    .agg(
        registros=("rbd", "size"),
        alumnos=("nalu_mate4b_rbd", "sum"),
        con_puntaje=("prom_mate4b_rbd", "count"),
    )
    .reset_index()
)
resumen_motivos["pct_del_bruto"] = resumen_motivos["registros"] / total_registros * 100
display(resumen_motivos.sort_values("registros", ascending=False))

# Distribución por código de marca
resumen_marcas = (
    df_raw.groupby("marca_mate4b_rbd", dropna=False)
    .agg(
        registros=("rbd", "size"),
        alumnos=("nalu_mate4b_rbd", "sum"),
        con_puntaje=("prom_mate4b_rbd", "count"),
    )
    .reset_index()
)
resumen_marcas["glosa"] = resumen_marcas["marca_mate4b_rbd"].map(OBS_PUNTAJE)
resumen_marcas["glosa"] = resumen_marcas["glosa"].fillna("Sin observación de exclusión")
resumen_marcas["pct_del_bruto"] = resumen_marcas["registros"] / total_registros * 100
display(resumen_marcas)

# Concentración territorial de casos excluidos que sí conservan puntaje.
# Esto permite detectar un posible sesgo territorial aun cuando el porcentaje global sea pequeño.
materialidad_territorial = (
    df_raw.loc[no_efectivos_con_puntaje]
    .groupby(["cod_reg_rbd", "nom_reg_rbd"], dropna=False)
    .agg(
        excluidos_con_puntaje=("rbd", "size"),
        puntaje_promedio_excluido=("prom_mate4b_rbd", "mean"),
    )
    .reset_index()
    .sort_values("excluidos_con_puntaje", ascending=False)
)
display(materialidad_territorial)

print(f"Conclusión de materialidad: {conclusion_materialidad}")

# ============================================================
# FILTRO: SOLO REGISTROS EFECTIVOS
# ============================================================

df_efectivo = df_raw.loc[condicion_efectiva].copy()

assert len(df_efectivo) == total_efectivos
assert len(df_efectivo) > 0, "Error: El proceso de filtrado descartó la totalidad de los datos."

print(f"Registros originales: {total_registros:,}")
print(f"Registros efectivos conservados: {len(df_efectivo):,}")
print(f"Registros no efectivos excluidos: {total_no_efectivos:,} ({pct_no_efectivos:.2f}%)")
print(
    f"No efectivos con puntaje: {cantidad_no_efectivos_con_puntaje:,} "
    f"({pct_no_efectivos_sobre_con_puntaje:.2f}% de los registros con puntaje)"
)


## 6. Enriquecimiento Taxonómico y Geográfico
Aplicación de las jerarquías territoriales y cualitativas mediante los mapeos previamente cargados. Se construyen cadenas compuestas de ubicación georeferenciada.
Los nombres se limpian antes de concatenar las ubicaciones, para que estas también queden normalizadas.

In [ ]:
# 1. Limpieza de textos
columnas_texto = [
    "nom_rbd", "nom_reg_rbd", "nom_pro_rbd", 
    "nom_com_rbd", "nom_deprov_rbd", "codigo_bbdd", "grado"
]

for col in columnas_texto:
    df_efectivo[col] = df_efectivo[col].astype("string").str.strip().replace("", pd.NA)

# Comprobar que los códigos disponibles tienen una etiqueta definida.
for columna, catalogo in [("cod_depe1", DEPENDENCIA_1), ("cod_depe2", DEPENDENCIA_2),
                          ("cod_grupo", GSE), ("cod_rural_rbd", RURALIDAD),
                          ("cod_reg_rbd", GEOGRAFIA_REGION)]:
    desconocidos = set(df_efectivo[columna].dropna()) - set(catalogo)
    assert not desconocidos, f"Códigos sin correspondencia en {columna}: {desconocidos}"

# Mapeo de variables categóricas
df_efectivo["dependencia_6_cat"] = df_efectivo["cod_depe1"].map(DEPENDENCIA_1)
df_efectivo["dependencia_4_cat"] = df_efectivo["cod_depe2"].map(DEPENDENCIA_2)
df_efectivo["grupo_socioeconomico"] = df_efectivo["cod_grupo"].map(GSE)
df_efectivo["ruralidad"] = df_efectivo["cod_rural_rbd"].map(RURALIDAD)

# Enriquecimiento geográfico
geo = df_efectivo["cod_reg_rbd"].map(GEOGRAFIA_REGION)

df_efectivo["Zona"] = geo.str[0]
df_efectivo["Macrozona"] = geo.str[1]
df_efectivo["Orden"] = geo.str[2].astype("Int64")
df_efectivo["pais"] = "Chile"

# Concatenación de ubicaciones jerárquicas
df_efectivo["ubicacion_region"] = df_efectivo["nom_reg_rbd"].astype("string") + ", Chile"
df_efectivo["ubicacion_provincia"] = (
    df_efectivo["nom_pro_rbd"].astype("string") + ", " + 
    df_efectivo["nom_reg_rbd"].astype("string") + ", Chile"
)
df_efectivo["ubicacion_comuna"] = (
    df_efectivo["nom_com_rbd"].astype("string") + ", " + 
    df_efectivo["nom_reg_rbd"].astype("string") + ", Chile"
)

print("Variables taxonómicas y geográficas generadas con éxito.")

## 7. Renombrado de Columnas
Traducción de nombres de columnas técnicos hacia nombres autoexplicativos. Los textos ya se normalizaron antes de construir las ubicaciones.

In [ ]:
# 2. Renombrar columnas a estándar final
df_efectivo = df_efectivo.rename(
    columns={
        "dvrbd": "dv_rbd",
        "nom_rbd": "nombre_establecimiento",
        "nom_reg_rbd": "region",
        "nom_pro_rbd": "provincia",
        "nom_com_rbd": "comuna",
        "nom_deprov_rbd": "deprov",
        "nalu_mate4b_rbd": "n_alumnos",
        "prom_mate4b_rbd": "puntaje_promedio",
        "palu_eda_ins_mate4b_rbd": "pct_insuficiente",
        "palu_eda_ele_mate4b_rbd": "pct_elemental",
        "palu_eda_ade_mate4b_rbd": "pct_adecuado",
        "noaplica": "no_aplica",
        "agno": "anio",
    }
)

# 3. Variables constantes complementarias
df_efectivo["asignatura"] = "Matematica"
df_efectivo["efectividad"] = "Efectiva"

print("Estandarización de columnas completada.")

## 8. Selección de Columnas y Validaciones de Calidad

Se genera la vista definitiva utilizando únicamente los registros clasificados como **Efectivos**.

Las columnas técnicas utilizadas para decidir la efectividad pueden eliminarse del dataset final porque su función ya quedó documentada y cuantificada en la etapa anterior. La trazabilidad se mantiene en el notebook mediante:

- `resumen_filtro`;
- `resumen_motivos`;
- `resumen_marcas`;
- `materialidad_territorial`;
- `conclusion_materialidad`.

### Interpretación de los datos eliminados
Los registros No Efectivos **no se eliminan por tener valores extremos ni para mejorar artificialmente el resultado**. Se excluyen porque no cumplen el criterio operacional de representatividad definido para el análisis.

Esto implica dos tipos de efecto:

1. **Pérdida de cobertura:** disminuye la cantidad de establecimientos disponibles para el análisis.
2. **Posible sesgo de selección:** si los No Efectivos se concentran en ciertas regiones, dependencias, grupos socioeconómicos o zonas rurales, la base final puede representar de forma desigual a la población original.

Por ese motivo la materialidad no se evalúa únicamente por el porcentaje total eliminado; también se revisa cuántos excluidos conservan puntaje y cómo se distribuyen territorialmente.


In [ ]:
COLUMNAS_FINALES = [
    "rbd", "dv_rbd", "nombre_establecimiento", "asignatura",
    "cod_reg_rbd", "region", "cod_pro_rbd", "provincia", "cod_com_rbd", "comuna", "deprov",
    "pais", "ubicacion_region", "ubicacion_provincia", "ubicacion_comuna",
    "Zona", "Macrozona", "Orden",
    "dependencia_6_cat", "dependencia_4_cat", "grupo_socioeconomico", "ruralidad",
    "n_alumnos", "puntaje_promedio",
    "pct_insuficiente", "pct_elemental", "pct_adecuado",
    "efectividad", "no_aplica", "codigo_bbdd", "fecha_bbdd", "grado", "anio"
]

df_final = df_efectivo[COLUMNAS_FINALES].copy()

# VALIDACIONES DE INTEGRIDAD (F2)
assert not df_final.empty, "ERROR: El dataset procesado quedó vacío."
assert df_final["asignatura"].eq("Matematica").all(), "ERROR: Hay registros con asignaturas distintas a Matemática."
assert df_final["efectividad"].eq("Efectiva").all(), "ERROR: Hay registros etiquetados como No Efectivos en la salida."
assert df_final["n_alumnos"].gt(0).all(), "ERROR: Hay registros con n_alumnos <= 0."
assert df_final.columns.duplicated().sum() == 0, "ERROR: Se detectaron columnas duplicadas en la salida."
assert df_final["rbd"].notna().all(), "ERROR: Existen registros con identificador RBD nulo."

assert df_final["rbd"].is_unique, "ERROR: Hay establecimientos repetidos."
assert df_final[["puntaje_promedio", "region", "provincia", "comuna"]].notna().all().all(), "ERROR: Faltan puntajes o datos territoriales para la comparacion."

# Códigos necesarios para agrupar por territorio sin depender del nombre.
assert df_final[["cod_reg_rbd", "cod_pro_rbd", "cod_com_rbd"]].notna().all().all(), "Faltan códigos territoriales."

# Los porcentajes pueden faltar; cuando existen deben ser coherentes.
columnas_pct = ["pct_insuficiente", "pct_elemental", "pct_adecuado"]
porcentajes = df_final[columnas_pct]
for col in columnas_pct:
    assert porcentajes[col].dropna().between(0, 100).all(), f"Porcentajes fuera de rango en {col}"
completos = porcentajes.dropna()
assert completos.sum(axis=1).sub(100).abs().le(0.15).all(), "Los porcentajes no suman 100 dentro del redondeo esperado."
print("Establecimientos sin los tres porcentajes:", int(porcentajes.isna().all(axis=1).sum()))
print("Establecimientos con porcentajes parcialmente disponibles:", int(porcentajes.isna().sum(axis=1).isin([1, 2]).sum()))

print("--- TODAS LAS VALIDACIONES DE CALIDAD FUE PASADAS CON ÉXITO ---")
print(f"Estructura Final: {df_final.shape[0]:,} filas x {df_final.shape[1]} columnas.")


## 9. Exportación del Dataset Procesado

Se exporta únicamente el dataset procesado en formato `.csv`, codificado como `UTF-8 con BOM` (`utf-8-sig`) para mantener compatibilidad con Excel, Power BI y Python.

El nombre del archivo incorpora la fecha y hora de ejecución con formato `AAAAMMDDHHMM`, permitiendo identificar de manera inequívoca cada corrida del pipeline.

Las estadísticas de filtrado y materialidad se muestran en el notebook, pero no generan archivos adicionales.


In [ ]:
df_final.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d"
)

# Verificación de ida y vuelta del archivo generado.
releido = pd.read_csv(OUTPUT_FILE, encoding="utf-8-sig")
releido["fecha_bbdd"] = pd.to_datetime(
    releido["fecha_bbdd"],
    format="%Y-%m-%d",
    errors="raise"
)

pd.testing.assert_frame_equal(
    releido,
    df_final.reset_index(drop=True),
    check_dtype=False
)

# ÚNICA SALIDA OPERATIVA DEL PIPELINE:
# 1) estadísticas en pantalla
# 2) archivo CSV procesado
print("=" * 70)
print("RESULTADO F2")
print("=" * 70)
print(f"Filas originales: {total_registros:,}")
print(f"Filas válidas / efectivas: {len(df_final):,}")
print(f"Filas no efectivas excluidas: {total_no_efectivos:,} ({pct_no_efectivos:.2f}%)")
print(
    f"No efectivas con puntaje: {cantidad_no_efectivos_con_puntaje:,} "
    f"({pct_no_efectivos_sobre_con_puntaje:.2f}% de los registros con puntaje)"
)
print(f"Materialidad: {conclusion_materialidad}")
print(f"Archivo generado: {OUTPUT_FILE.name}")


## 10. Conclusión de la Fase 2

La limpieza realizada en esta fase distingue entre **eliminar datos por problemas de calidad** y **excluir observaciones que no cumplen condiciones de representatividad**.

En la ejecución registrada previamente en este notebook, el archivo bruto contenía 7.143 establecimientos y la salida efectiva conservaba 6.524, por lo que 619 registros quedaban fuera del análisis. Esto representa aproximadamente **8,67% del archivo original**, por lo que la exclusión es **material en términos de cobertura**.

Sin embargo, el diagnóstico previo también mostraba que solo 55 registros excluidos conservaban puntaje. Eso equivale aproximadamente a **0,77% del archivo bruto** y cerca de **0,84% de los registros que poseen puntaje**. Por lo tanto, el conjunto de No Efectivos puede ser material por cantidad de establecimientos, pero su materialidad directa sobre la disponibilidad de puntajes es considerablemente menor.

Esta conclusión no significa que los 55 casos puedan ignorarse. Si se concentran en una región, dependencia, grupo socioeconómico o zona rural específica, podrían introducir un sesgo de cobertura. Por eso el pipeline incorpora una tabla territorial específica para esos casos y una prueba de sensibilidad del promedio.

### Decisión metodológica
Los registros No Efectivos se mantienen fuera del dataset final porque no cumplen el criterio operacional de efectividad. No se imputan sus resultados, no se reemplazan por cero y no se utilizan para completar artificialmente la muestra. La decisión queda documentada mediante indicadores reproducibles antes de ejecutar el filtro.

El dataset final queda preparado para la Fase 3, manteniendo una separación clara entre:

- datos válidos para modelamiento;
- registros descartados;
- causas del descarte;
- magnitud del descarte;
- posible materialidad del descarte.
